<a href="https://colab.research.google.com/github/ghadirchhade/Master-Thesis/blob/main/baseline1_new5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [2]:
!pip install -q torch torchvision

In [3]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA is available: True


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# facebook/sam3 is gated on the Hub -> you must accept the license at
# https://huggingface.co/facebook/sam3 with the account whose token you use below.
!pip install -q -U transformers accelerate huggingface_hub supervision

from huggingface_hub import login
login()  # paste your HF token (needs access to facebook/sam3)

print("Transformers SAM3 dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 15.6 MB/s eta 0:00:00
Transformers SAM3 dependencies installed.


In [6]:
import os, glob, csv, time
import numpy as np
import pandas as pd
import gc
import torch
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import supervision as sv
import matplotlib.patches as patches
from transformers import Sam3Model, Sam3Processor

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

sam3_model = Sam3Model.from_pretrained("facebook/sam3", device_map="auto")
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("HF transformers SAM3 model + processor loaded.")
print("Model device:", next(sam3_model.parameters()).device)

config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

HF transformers SAM3 model + processor loaded.
Model device: cuda:0


In [8]:
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0

In [9]:
#   USE_PRESENCE_GATE      -- best-effort score reweighting using
#                              SAM3's presence head (see caveat below)
#   ADD_GLOBAL_CONTEXT_PASS -- optional extra pass on a downsampled
#                              whole image, merged with tile results
# Both default to values I'd actually recommend: presence gating ON
# (it's free if it doesn't fire, harmless if it does), global pass
# OFF (see caveat in run_sam3_pipeline -- it's more likely to hurt
# small-object recall than help, given your image size).

EXPERIMENT_NAME = "E02_2"   # e.g. "E02_2" (Exp1), "<exp2_name>", "<exp3_name>"
N_EXEMPLARS     = 3         # 3 for Exp1/Exp2, 1 for Exp3
USE_TILING      = True      # True for Exp1/Exp3, False for Exp2

# SAM3 / tiling params
# TILE_SIZE = 2000   # comfortably > max plant width (706px), leaves ~65% of the tile as context
# OVERLAP   = 800     # > max plant width (706px), so any plant is fully captured whole in some tile
TILE_SIZE = 1536
OVERLAP = 384 #25%
THRESHOLD = 0.3
IOU_THRESHOLD = 0.5
NMS_IOU_THRESHOLD = 0.5

#USE_PRESENCE_GATE = True          # multiply detection scores by SAM3's presence score, per tile
ADD_GLOBAL_CONTEXT_PASS = True   # extra downsampled whole-image pass -- OFF by default, see caveat
GLOBAL_DOWNSCALE = 2              # how much to shrink the image for the global pass, if enabled

# Batched inference settings
BATCH_SIZE = 1    # how many tiles get sent to the GPU in one model call -- lower if you hit OOM, raise if GPU memory sits idle
USE_FP16   = True  # run inference in half precision -- roughly halves memory use, usually near-zero accuracy cost

KEEP_MASKS = False   # True only if you'll actually use masks later (visualization, etc.)

RNG = np.random.default_rng(42)  # fixed seed -> reproducible exemplar sampling

In [10]:
# Output CSV (append-safe: survives Colab disconnects)
OUTPUT_CSV = f"/content/drive/MyDrive/master_thesis/results/results4_{EXPERIMENT_NAME}.csv"
CSV_COLUMNS = ["experiment_name", "image_ID", "anchor_idx", "Prompt_ID", "Prompt_Type", "mAP50",
               "precision", "recall", "IoU1", "IoU2"]

PROMPT_TYPE = "multiple" if N_EXEMPLARS > 1 else "single"

In [11]:
def load_yolo_boxes(label_path, img_width, img_height, class_id=0):
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls = int(parts[0])
            if cls != class_id:
                continue
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc, bw, bh = xc * img_width, yc * img_height, bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32)

In [12]:
def format_prompt_id(exemplar_indices: list) -> str:
    """
    Single box (Exp3)   -> "5"
    Multiple boxes (Exp1/Exp2) -> "5+12+3"  (anchor first, then the sampled others,
    order preserved so you can always tell which one was the anchor: it's the
    first number in the string)
    """
    return "+".join(str(i) for i in exemplar_indices)

In [13]:
def find_label_path(image_filename_no_ext, folder_name):
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")  # <- change here if needed
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")                    # <- and here
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None

In [14]:
#Discover every image across all folders in AGS_MULTI_RUMEX
image_records = []  # (folder, image_path, label_path, image_id)
VALID_EXT = (".jpg", ".jpeg", ".png")

for folder in sorted(os.listdir(IMAGES_ROOT)):
    folder_path = os.path.join(IMAGES_ROOT, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith(VALID_EXT):
            continue
        name_no_ext = os.path.splitext(fname)[0]
        label_path = find_label_path(name_no_ext, folder)
        image_id = f"{folder}/{name_no_ext}"
        image_records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
        missing_labels = [r for r in image_records if r[2] is None]

print(f"Discovered {len(image_records)} images across {len(set(r[0] for r in image_records))} folders.")
if missing_labels:
    print(f"WARNING: {len(missing_labels)} images have no matching label file (will be skipped). "
          f"First few: {[r[3] for r in missing_labels[:5]]}")

Discovered 179 images across 15 folders.


In [15]:
# # ============================================================
# # NEW CELL: how big are the Rumex plants actually, in pixels?
# #
# # In plain English: this cell looks at every ground-truth box in
# # your whole dataset and measures how wide/tall it is in pixels.
# # Right now you're guessing whether TILE_SIZE=1000 is a good number.
# # This cell replaces the guess with real numbers from your own data,
# # so you know if plants are, say, 40px wide (in which case 1000px
# # tiles are plenty) or 300px wide (in which case tiles that small
# # would be chopping plants in half all the time, hurting detection).
# #
# # Run this once, look at the printed numbers, THEN decide whether
# # TILE_SIZE needs to change. Don't change TILE_SIZE before running
# # this.
# # ============================================================
# box_widths, box_heights = [], []

# for folder, image_path, label_path, image_id in image_records:
#     if label_path is None:
#         continue
#     with Image.open(image_path) as im:
#         w, h = im.size
#     boxes = load_yolo_boxes(label_path, w, h, class_id=RUMEX_CLASS_ID)
#     if len(boxes) == 0:
#         continue
#     box_widths.extend((boxes[:, 2] - boxes[:, 0]).tolist())
#     box_heights.extend((boxes[:, 3] - boxes[:, 1]).tolist())

# box_widths = np.array(box_widths)
# box_heights = np.array(box_heights)

# print(f"n GT boxes: {len(box_widths)}")
# print(f"width  (px): median={np.median(box_widths):.1f}  p90={np.percentile(box_widths,90):.1f}  max={box_widths.max():.1f}")
# print(f"height (px): median={np.median(box_heights):.1f}  p90={np.percentile(box_heights,90):.1f}  max={box_heights.max():.1f}")

# # Simple guideline: divide your p90 width by TILE_SIZE. If that's under
# # ~10%, your current TILE_SIZE is fine. If it's much higher than that,
# # consider raising TILE_SIZE so plants aren't a huge chunk of every tile.

In [16]:
def select_exemplar_indices(n_gt: int, anchor_idx: int, n_exemplars: int, image_id: str) -> list:
    """
    Builds the exemplar index set for one run:
      - always includes the anchor box (the one this row is "about")
      - fills the rest (n_exemplars - 1 slots) with OTHER GT boxes from the
        same image, sampled without replacement
      - if the image doesn't have enough other GT boxes, just uses whatever
        is available (no duplication, no crash)
    """
    ## each GT bbox gets a chance to act as the "anchor"
    ## prompt, while the remaining exemplars are randomly sampled from the same image.
    seed = abs(hash((image_id, anchor_idx))) % (2**32)
    rng = np.random.default_rng(seed)
    others = [i for i in range(n_gt) if i != anchor_idx]
    n_others_needed = min(n_exemplars - 1, len(others))   # min(1 - 1, ...) = min(0, ...) = 0
    if n_others_needed > 0: #in case of several input bboxes
        chosen_others = list(rng.choice(others, size=n_others_needed, replace=False))
    else: #in case of 1 GT as a bbox (1 anchor per run)=>no random sampling
        chosen_others = []                                # <-- always hits this branch
    return [anchor_idx] + chosen_others                    # -> [anchor_idx]

In [17]:
def get_local_background_patch(tile_img: Image.Image, patch_w: int, patch_h: int) -> Image.Image:
    """
    Instead of a flat white block, sample REAL texture from the tile itself to use as
    the strip's background. Why this matters:
      - Color balance, brightness, and grain automatically match this tile's own
        lighting conditions (which can vary tile-to-tile across a large drone photo).
      - There's no artificial white-vs-photo boundary for the vision transformer's
        self-attention to latch onto as a spurious "edge feature".
      - The exemplar's ROI-pooled feature (computed AFTER the image encoder has already
        mixed in neighboring context via attention) ends up looking like "a plant in
        grass" instead of "a plant in a void" -- much closer to a real target instance.

    We grab the top-left corner of the tile from target img (an area that won't be duplicated, since
    the strip sits ABOVE the tile in the final canvas, not overlapping it).
    """
    sample = tile_img.crop((0, 0, min(patch_w, tile_img.width), min(patch_h, tile_img.height)))
    if sample.size != (patch_w, patch_h):
        sample = sample.resize((patch_w, patch_h))
    return sample


def make_feather_mask(size, feather_width):
    w, h = size
    mask = np.full((h, w), 255.0, dtype=np.float32)

    effective_feather = min(feather_width, h // 2, w // 2)
    if effective_feather >= 1:
        for i in range(effective_feather):
            alpha = 255.0 * (i + 1) / effective_feather
            mask[i, :] = np.minimum(mask[i, :], alpha)
            mask[h - 1 - i, :] = np.minimum(mask[h - 1 - i, :], alpha)
            mask[:, i] = np.minimum(mask[:, i], alpha)
            mask[:, w - 1 - i] = np.minimum(mask[:, w - 1 - i], alpha)

    return Image.fromarray(mask.astype(np.uint8), mode="L")  # <-- return a PIL Image, not ndarray

In [18]:
def compose_tile_with_exemplars(tile_img: Image.Image, crop_images: list,
                                 margin: int = 6, feather_width: int = 8):
    """
    Builds the small composed image actually sent to SAM3, for ONE tile, with
    MULTIPLE exemplar crops placed side-by-side in a strip above the tile.

    Returns:
        composed  -- the image to feed to SAM3
        crop_boxes -- list of [x1,y1,x2,y2], one per exemplar, in composed-image coords
        offset    -- (dx, dy) to convert composed target-region coords back
                     to this TILE's own coordinate system
    """
    n = len(crop_images)
    strip_h = max(c.height for c in crop_images) + 2 * margin
    strip_content_w = sum(c.width for c in crop_images) + margin * (n + 1)
    canvas_w = max(tile_img.width, strip_content_w)
    canvas_h = strip_h + tile_img.height

    strip_bg = get_local_background_patch(tile_img, canvas_w, strip_h)

    composed = Image.new("RGB", (canvas_w, canvas_h))
    composed.paste(strip_bg, (0, 0))
    offset = (0, strip_h)
    composed.paste(tile_img, offset)

    crop_boxes = []
    cursor_x = margin
    for crop_image in crop_images:
        feather_mask = make_feather_mask(crop_image.size, feather_width=feather_width)
        paste_xy = (cursor_x, margin)
        composed.paste(crop_image, paste_xy, feather_mask)

        crop_boxes.append([
            paste_xy[0], paste_xy[1],
            paste_xy[0] + crop_image.width, paste_xy[1] + crop_image.height,
        ])
        cursor_x += crop_image.width + margin

    return composed, crop_boxes, offset

In [19]:
def tile_bboxes(img_w: int, img_h: int, tile_size: int, overlap: int):
    """
    Generates (x1,y1,x2,y2) windows covering the whole image, with overlap so plants
    sitting right on a tile boundary aren't missed or half-cut in every window.
    """
    step = tile_size - overlap
    tiles = []
    for y in range(0, img_h, step):
        for x in range(0, img_w, step):
            x2 = min(x + tile_size, img_w)
            y2 = min(y + tile_size, img_h)
            x1 = max(0, x2 - tile_size)
            y1 = max(0, y2 - tile_size)
            tiles.append((x1, y1, x2, y2))
    return list(dict.fromkeys(tiles))

In [20]:
# MODIFIED CELL: keep_only_target_region_detections
# Only change: masks are explicitly cast to a boolean array before
# being stored. If they were already coming back as float32
# (common if the model outputs raw sigmoid probabilities before
# any threshold is applied), this alone can cut mask memory by
# ~4-8x. If they're already bool, this is a no-op.

def keep_only_target_region_detections(boxes, scores, masks, offset, y_tolerance=5):
    """
    Drop any detection sitting inside the padding strip (that's a pasted exemplar
    being re-detected, not a real find), keep only detections that fall within the
    tile's real-content region, and remap their coordinates back to that tile's own
    coordinate system (undo the paste offset).
    """
    dx, dy = offset
    kept_boxes, kept_scores, kept_masks = [], [], []

    for box, score, mask in zip(boxes, scores, masks):
        x1, y1, x2, y2 = box.tolist() if torch.is_tensor(box) else box
        if y1 >= dy - y_tolerance:
            remapped_box = [x1 - dx, max(y1 - dy, 0), x2 - dx, y2 - dy]
            kept_boxes.append(remapped_box)
            kept_scores.append(score)
            mask_np = mask.cpu().numpy() if torch.is_tensor(mask) else mask
            mask_np = mask_np[dy:, dx:] if dx or dy else mask_np
            kept_masks.append(mask_np.astype(np.bool_))  # was implicitly float before
    return kept_boxes, kept_scores, kept_masks

In [21]:
def filter_implausible_boxes(boxes, scores, masks, tile_w, tile_h,
                              min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5):
    """
    Drops detections unlikely to be real single Rumex plants:
      - box area implausibly large relative to the tile (likely background clutter,
        not a single plant)
      - predicted mask fills too little of its own box (weak/noisy detection rather
        than a confident, well-localized instance)
      - degenerate boxes clipped to (near) zero size at the tile edges
    `edge_margin` is used to tolerate boxes that legitimately touch a tile border
    (real plants often do, since tiles overlap and neighboring tiles will pick up
    the rest of the same plant) rather than penalizing them outright.
    """
    kept_boxes, kept_scores, kept_masks = [], [], []
    tile_area = tile_w * tile_h

    for box, score, mask in zip(boxes, scores, masks):
        x1, y1, x2, y2 = box
        box_w, box_h = x2 - x1, y2 - y1
        if box_w <= edge_margin or box_h <= edge_margin:
            continue

        box_area = box_w * box_h
        if box_area / tile_area > max_area_fraction:
            continue

        mask_np = mask.cpu().numpy() if torch.is_tensor(mask) else mask
        x1c, y1c = int(max(0, x1)), int(max(0, y1))
        x2c, y2c = int(min(mask_np.shape[1], x2)), int(min(mask_np.shape[0], y2))
        if x2c <= x1c or y2c <= y1c:
            continue

        region = mask_np[y1c:y2c, x1c:x2c]
        fill_ratio = (region > 0.5).mean() if region.size > 0 else 0.0
        if fill_ratio < min_fill_ratio:
            continue

        kept_boxes.append(box)
        kept_scores.append(score)
        kept_masks.append(None)

    return kept_boxes, kept_scores, kept_masks

In [22]:
# # CHANGE 2: batched + cached tile inference, ported from code #1.
# def run_sam3_on_tiles_batched(composed_tiles, crop_boxes, offset, threshold=0.3,
#                                 batch_size=4, use_fp16=True):
#     """
#     composed_tiles: list of PIL images, all the SAME size (one per spatial tile),
#                     for ONE anchor's exemplar set.
#     crop_boxes/offset: identical for every tile in this anchor's run (exemplar
#                         strip position never changes tile-to-tile).

#     Returns a list of (kept_boxes, kept_scores, kept_masks) in the same order
#     as composed_tiles.

#     OOM-safe: if a batch fails with CUDA OOM, it's automatically split into
#     smaller batches (halved) and retried, so you never crash the whole loop.
#     """
#     all_results = [None] * len(composed_tiles)

#     def process_batch(indices):
#         batch = [composed_tiles[i] for i in indices]
#         b = len(batch)
#         try:
#             inputs = sam3_processor(
#                 images=batch,
#                 input_boxes=[[[float(c) for c in box] for box in crop_boxes]] * b,
#                 input_boxes_labels=[[1] * len(crop_boxes)] * b,
#                 return_tensors="pt",
#             ).to(sam3_model.device)

#             with torch.inference_mode():
#                 if use_fp16:
#                     with torch.autocast(device_type="cuda", dtype=torch.float16):
#                         outputs = sam3_model(**inputs)
#                 else:
#                     outputs = sam3_model(**inputs)

#             batch_results = sam3_processor.post_process_instance_segmentation(
#                 outputs, threshold=threshold, mask_threshold=0.4,
#                 target_sizes=inputs.get("original_sizes").tolist(),
#             )

#             for idx, res in zip(indices, batch_results):
#                 all_results[idx] = keep_only_target_region_detections(
#                     res["boxes"], res["scores"], res["masks"], offset
#                 )

#             del inputs, outputs, batch_results
#             torch.cuda.empty_cache()

#         except torch.cuda.OutOfMemoryError:
#             torch.cuda.empty_cache()
#             if b == 1:
#                 print(f"    WARNING: OOM on a single tile, skipping it (threshold/tile too large for T4).")
#                 all_results[indices[0]] = ([], [], [])
#                 return
#             mid = b // 2
#             print(f"    OOM at batch_size={b}, splitting into {mid} + {b - mid} and retrying...")
#             process_batch(indices[:mid])
#             process_batch(indices[mid:])

#     for start in range(0, len(composed_tiles), batch_size):
#         chunk = list(range(start, min(start + batch_size, len(composed_tiles))))
#         process_batch(chunk)

#     return all_results

In [23]:
# ============================================================
# MODIFIED CELL: run_sam3_on_tiles_batched
# Added: presence-token gating, applied per-item inside each batch.
#
# Plain English: after the model looks at a tile, it can also output
# a "presence" score -- its own estimate of whether the thing you're
# looking for (a Rumex plant matching the exemplar crops) is present
# ANYWHERE in that tile at all, separate from where exactly it might
# be. We multiply every detection's confidence in that tile by this
# presence score. A tile of pure empty grass that the model is only
# 20% confident contains anything real will have its detections
# knocked down (0.6 confidence x 0.2 presence = 0.12, likely dropped
# below threshold). A tile the model is 95% sure has a real plant
# keeps its detections almost untouched.
#
# CAVEAT (same as before): presence_logits comes back with shape
# (batch_size, 1) -- one score per tile in the batch. It's documented
# around TEXT-conditioned prompts; in box-only exemplar mode (no
# text=, which is what this pipeline uses) some transformers versions
# may leave it as None. The code checks for that and just skips
# gating if so -- behavior falls back to exactly what you had before,
# nothing crashes either way. Print a few presence_prob values the
# first time you run this to confirm it's actually firing for you.
# ============================================================
def run_sam3_on_tiles_batched(composed_tiles, crop_boxes, offset, threshold=0.3,
                                batch_size=4, use_fp16=True, use_presence_gate=True):
    all_results = [None] * len(composed_tiles)

    def process_batch(indices):
        batch = [composed_tiles[i] for i in indices]
        b = len(batch)
        try:
            inputs = sam3_processor(
                images=batch,
                input_boxes=[[[float(c) for c in box] for box in crop_boxes]] * b,
                input_boxes_labels=[[1] * len(crop_boxes)] * b,
                return_tensors="pt",
            ).to(sam3_model.device)

            with torch.inference_mode():
                if use_fp16:
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        outputs = sam3_model(**inputs)
                else:
                    outputs = sam3_model(**inputs)

            # # --- presence scores for this batch, one per tile (best-effort) ---
            # presence_probs = None
            # if use_presence_gate and getattr(outputs, "presence_logits", None) is not None:
            #     # shape (b, 1) -> squeeze to a flat list of b floats, one per tile
            #     presence_probs = torch.sigmoid(outputs.presence_logits).squeeze(-1).tolist()
            #     if isinstance(presence_probs, float):  # b == 1 case, squeeze can over-flatten
            #         presence_probs = [presence_probs]

            batch_results = sam3_processor.post_process_instance_segmentation(
                outputs, threshold=threshold, mask_threshold=0.4,
                target_sizes=inputs.get("original_sizes").tolist(),
            )

            # # --- apply presence gating per tile, before remapping coordinates ---
            # if presence_probs is not None:
            #     for res, p in zip(batch_results, presence_probs):
            #         if len(res["scores"]) == 0:
            #             continue
            #         rescaled = [s * p for s in res["scores"]]
            #         keep_idx = [i for i, s in enumerate(rescaled) if s >= threshold]
            #         res["boxes"] = [res["boxes"][i] for i in keep_idx]
            #         res["scores"] = [rescaled[i] for i in keep_idx]
            #         res["masks"] = [res["masks"][i] for i in keep_idx]

            for idx, res in zip(indices, batch_results):
                all_results[idx] = keep_only_target_region_detections(
                    res["boxes"], res["scores"], res["masks"], offset
                )

            del inputs, outputs, batch_results
            torch.cuda.empty_cache()

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if b == 1:
                print(f"    WARNING: OOM on a single tile, skipping it (threshold/tile too large for T4).")
                all_results[indices[0]] = ([], [], [])
                return
            mid = b // 2
            print(f"    OOM at batch_size={b}, splitting into {mid} + {b - mid} and retrying...")
            process_batch(indices[:mid])
            process_batch(indices[mid:])

    for start in range(0, len(composed_tiles), batch_size):
        chunk = list(range(start, min(start + batch_size, len(composed_tiles))))
        process_batch(chunk)

    return all_results

In [24]:
def nms_merge(boxes, scores, masks, iou_thresh: float = NMS_IOU_THRESHOLD):
    """
    Because tiles overlap, the same real plant can get detected in two neighboring
    tiles. Standard IoU-based Non-Max Suppression collapses duplicates down to the
    single highest-confidence box -- carries the matching mask through too.
    """
    if not boxes:
        return [], [], []

    boxes_t = torch.tensor(boxes, dtype=torch.float32)
    scores_t = torch.tensor([s.item() if torch.is_tensor(s) else s for s in scores])
    order = scores_t.argsort(descending=True)
    keep = []

    while order.numel() > 0:
        i = order[0].item()
        keep.append(i)
        if order.numel() == 1:
            break
        rest = order[1:]
        xx1 = torch.maximum(boxes_t[i, 0], boxes_t[rest, 0])
        yy1 = torch.maximum(boxes_t[i, 1], boxes_t[rest, 1])
        xx2 = torch.minimum(boxes_t[i, 2], boxes_t[rest, 2])
        yy2 = torch.minimum(boxes_t[i, 3], boxes_t[rest, 3])
        inter = (xx2 - xx1).clamp(0) * (yy2 - yy1).clamp(0)
        area_i = (boxes_t[i, 2] - boxes_t[i, 0]) * (boxes_t[i, 3] - boxes_t[i, 1])
        area_r = (boxes_t[rest, 2] - boxes_t[rest, 0]) * (boxes_t[rest, 3] - boxes_t[rest, 1])
        iou = inter / (area_i + area_r - inter + 1e-6)
        order = rest[iou <= iou_thresh]

    kept_boxes = [boxes[i] for i in keep]
    kept_scores = [scores[i] for i in keep]
    kept_masks = [masks[i] for i in keep]
    return kept_boxes, kept_scores, kept_masks

In [25]:
# # Unified SAM3 pipeline -- tiled (batched + cached) vs whole-image.
# # filter_implausible_boxes() call sites removed (change #3).
# def run_sam3_pipeline(image, exemplar_crops, use_tiling, tile_size=768, overlap=150,
#                        threshold=0.3, cached_tiles=None, batch_size=BATCH_SIZE,
#                        nms_iou_thresh=NMS_IOU_THRESHOLD):
#     all_boxes, all_scores, all_masks = [], [], []

#     if use_tiling:
#         # CHANGE 2: reuse cached tile crops if the caller already computed them
#         # for this image (built once per image, outside the anchor loop).
#         tiles = cached_tiles if cached_tiles is not None else [
#             (x1, y1, x2, y2, image.crop((x1, y1, x2, y2)))
#             for (x1, y1, x2, y2) in tile_bboxes(image.width, image.height, tile_size, overlap)
#         ]

#         for start in range(0, len(tiles), batch_size):
#             chunk = tiles[start:start + batch_size]
#             composed_tiles, crop_boxes, offset = [], None, None
#             for (x1, y1, x2, y2, tile) in chunk:
#                 ct, cb, off = compose_tile_with_exemplars(tile, exemplar_crops, margin=6, feather_width=8)
#                 composed_tiles.append(ct)
#                 crop_boxes, offset = cb, off

#             batch_results = run_sam3_on_tiles_batched(
#                 composed_tiles, crop_boxes, offset, threshold=threshold, batch_size=batch_size
#             )

#             for (x1, y1, x2, y2, _), (boxes, scores, masks) in zip(chunk, batch_results):
#                             tile_w, tile_h = x2 - x1, y2 - y1
#                             boxes, scores, masks = filter_implausible_boxes(
#                                 boxes, scores, masks, tile_w, tile_h,
#                                 min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5,
#                             )
#                             for b, s, m in zip(boxes, scores, masks):
#                               all_boxes.append([b[0] + x1, b[1] + y1, b[2] + x1, b[3] + y1])
#                               all_scores.append(s.item() if torch.is_tensor(s) else s)
#                               all_masks.append((m, x1, y1))

#             del composed_tiles
#         torch.cuda.empty_cache()

#     else:
#         composed, crop_boxes, offset = compose_tile_with_exemplars(image, exemplar_crops, margin=6, feather_width=8)
#         results = run_sam3_on_tiles_batched([composed], crop_boxes, offset, threshold=threshold, batch_size=1)
#         boxes, scores, masks = results[0]
#         boxes, scores, masks = filter_implausible_boxes(
#             boxes, scores, masks, image.width, image.height,
#             min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5,
#         )
#         for b, s, m in zip(boxes, scores, masks):
#             all_boxes.append([b[0], b[1], b[2], b[3]])
#             all_scores.append(s.item() if torch.is_tensor(s) else s)
#             all_masks.append((m, 0, 0))

#     # NMS is applied once on the entire original image, after all detections
#     # from all tiles have been collected (not separately inside each tile).
#     final_boxes, final_scores, final_masks = nms_merge(all_boxes, all_scores, all_masks, iou_thresh=nms_iou_thresh)
#     return final_boxes, final_scores, final_masks

In [26]:
# ============================================================
# MODIFIED CELL: run_sam3_pipeline
# Added: an optional extra pass on a downsampled copy of the WHOLE
# image (no tiling), whose detections get scaled back up and merged
# in with the tiled detections via NMS.
#
# REPEATING THE CAVEAT FROM BEFORE, because it matters here: SAM3's
# own image processor resizes whatever you feed it to ~1024px on the
# longest side before the model ever sees it -- that happens whether
# you hand it a 1000px tile or the full 8192px image. So shrinking
# your image by GLOBAL_DOWNSCALE=2 (giving a 4096x2730 image) doesn't
# preserve detail -- SAM3 crunches it down to ~1024px anyway, i.e.
# ANOTHER ~4x shrink on top of the one you already did. A Rumex plant
# that's ~50px wide in the original image would end up only a few
# pixels wide by the time the model actually processes it in this
# pass -- likely too small to detect. This is why ADD_GLOBAL_CONTEXT_PASS
# defaults to False. Test it on a small subset (10-15 images, compare
# mAP50 with it on vs off) before trusting it on a full experiment run.
# ============================================================
def run_sam3_pipeline(image, exemplar_crops, use_tiling, tile_size=1000, overlap=150,
                       threshold=0.3, cached_tiles=None, batch_size=BATCH_SIZE,
                       nms_iou_thresh=NMS_IOU_THRESHOLD, use_fp16=USE_FP16,
                       add_global_pass=ADD_GLOBAL_CONTEXT_PASS, global_downscale=GLOBAL_DOWNSCALE,
                       keep_masks=KEEP_MASKS):
    all_boxes, all_scores, all_masks = [], [], []

    if use_tiling:
        tiles = cached_tiles if cached_tiles is not None else [
            (x1, y1, x2, y2, image.crop((x1, y1, x2, y2)))
            for (x1, y1, x2, y2) in tile_bboxes(image.width, image.height, tile_size, overlap)
        ]

        for start in range(0, len(tiles), batch_size):
            chunk = tiles[start:start + batch_size]
            composed_tiles = []
            crop_boxes, offset = None, None
            for (x1, y1, x2, y2, tile) in chunk:
                ct, cb, off = compose_tile_with_exemplars(tile, exemplar_crops, margin=6, feather_width=8)
                composed_tiles.append(ct)
                crop_boxes, offset = cb, off

            batch_results = run_sam3_on_tiles_batched(
                composed_tiles, crop_boxes, offset,
                threshold=threshold, batch_size=batch_size, use_fp16=use_fp16,
            )

            for (x1, y1, x2, y2, _), (boxes, scores, masks) in zip(chunk, batch_results):
                tile_w, tile_h = x2 - x1, y2 - y1
                boxes, scores, masks = filter_implausible_boxes(
                    boxes, scores, masks, tile_w, tile_h,
                    min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5,
                )
                for b, s, m in zip(boxes, scores, masks):
                    all_boxes.append([b[0] + x1, b[1] + y1, b[2] + x1, b[3] + y1])
                    all_scores.append(s.item() if torch.is_tensor(s) else s)
                    all_masks.append((m, x1, y1) if keep_masks else None)

            for ct in composed_tiles:
                ct.close()
            del composed_tiles

        torch.cuda.empty_cache()

    else:
        composed, crop_boxes, offset = compose_tile_with_exemplars(image, exemplar_crops, margin=6, feather_width=8)
        results = run_sam3_on_tiles_batched(
            [composed], crop_boxes, offset, threshold=threshold, batch_size=1,
            use_fp16=use_fp16,
        )
        boxes, scores, masks = results[0]
        boxes, scores, masks = filter_implausible_boxes(
            boxes, scores, masks, image.width, image.height,
            min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5,
        )
        for b, s, m in zip(boxes, scores, masks):
            all_boxes.append([b[0], b[1], b[2], b[3]])
            all_scores.append(s.item() if torch.is_tensor(s) else s)
            all_masks.append((m, 0, 0))

        composed.close()


    # ---- Optional global downsampled context pass (off by default) ----
    if add_global_pass:
        small_img = image.resize((image.width // global_downscale, image.height // global_downscale))
        # single-image, non-tiled call -- reuses the whole-image branch logic above
        g_composed, g_crop_boxes, g_offset = compose_tile_with_exemplars(
            small_img, exemplar_crops, margin=6, feather_width=8
        )
        g_results = run_sam3_on_tiles_batched(
            [g_composed], g_crop_boxes, g_offset, threshold=threshold, batch_size=1,
            use_fp16=use_fp16
        )
        g_boxes, g_scores, g_masks = g_results[0]
        g_boxes, g_scores, g_masks = filter_implausible_boxes(
            g_boxes, g_scores, g_masks, small_img.width, small_img.height,
            min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5,
        )
        # scale detections from the small image back up to full-image coordinates
        scale = image.width / small_img.width
        for b, s, m in zip(g_boxes, g_scores, g_masks):
            all_boxes.append([b[0]*scale, b[1]*scale, b[2]*scale, b[3]*scale])
            all_scores.append(s.item() if torch.is_tensor(s) else s)
            all_masks.append((m, 0, 0))  # note: mask is still at small_img resolution, not rescaled

        g_composed.close()
        small_img.close()
        gc.collect()
        torch.cuda.empty_cache()

    # NMS runs once at the end across everything -- tiled detections AND
    # (if enabled) the global-pass detections -- to remove duplicates.
    final_boxes, final_scores, final_masks = nms_merge(all_boxes, all_scores, all_masks, iou_thresh=nms_iou_thresh)
    return final_boxes, final_scores, final_masks

In [27]:
# Metrics helper -- refactored from Exp1's evaluation cell
# Returns mAP50, precision, recall, IoU1 (matched-only), IoU2 (over all GT)
import supervision as sv
from supervision.metrics import MeanAveragePrecision

def compute_iou_matrix(boxes1, boxes2):
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)))
    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])
    inter_w = np.clip(x2 - x1, 0, None)
    inter_h = np.clip(y2 - y1, 0, None)
    inter_area = inter_w * inter_h
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union_area = area1[:, None] + area2[None, :] - inter_area
    return np.where(union_area > 0, inter_area / union_area, 0.0)


def compute_detection_metrics(final_boxes, final_scores, gt_boxes, iou_threshold: float = 0.5) -> dict:
    """
    final_boxes/final_scores: predictions for ONE run (one image, one exemplar set)
    gt_boxes: ALL GT boxes for that image (numpy array, xyxy)
    Mirrors exactly the metric logic from Exp1's evaluation cell.
    """
    pred_boxes_np = np.array(final_boxes, dtype=np.float32) if final_boxes else np.zeros((0, 4), dtype=np.float32)
    pred_scores_np = np.array(final_scores, dtype=np.float32) if final_scores else np.zeros((0,), dtype=np.float32)

    # --- mAP50 via supervision ---
    pred_detections = sv.Detections(
        xyxy=pred_boxes_np,
        confidence=pred_scores_np,
        class_id=np.zeros(len(pred_scores_np), dtype=int),
    )
    gt_detections = sv.Detections(
        xyxy=gt_boxes,
        class_id=np.zeros(len(gt_boxes), dtype=int),
    )
    try:
        map_metric = MeanAveragePrecision()
        result = map_metric.update([pred_detections], [gt_detections]).compute()
        map50 = float(result.map50)
    except Exception:
        # supervision can raise/return NaN on degenerate cases (0 preds & 0 GT etc.)
        map50 = 0.0 if len(gt_boxes) > 0 else float("nan")

    #  precision / recall / IoU via greedy matching
    num_preds, num_gt = len(pred_boxes_np), len(gt_boxes)
    iou_matrix = compute_iou_matrix(pred_boxes_np, gt_boxes)

    matched_gt = set()
    true_positives = 0
    matched_ious = []
    pred_order = np.argsort(-pred_scores_np) if num_preds > 0 else []

    for pred_idx in pred_order:
        if num_gt == 0:
            break
        best_gt_idx = np.argmax(iou_matrix[pred_idx])
        best_iou = iou_matrix[pred_idx, best_gt_idx]
        if best_iou >= iou_threshold and best_gt_idx not in matched_gt:
            matched_gt.add(best_gt_idx)
            true_positives += 1
            matched_ious.append(best_iou)

    precision = true_positives / num_preds if num_preds > 0 else 0.0
    recall = true_positives / num_gt if num_gt > 0 else 0.0
    iou1_matched_only = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2_over_all_gt = float(np.sum(matched_ious) / num_gt) if num_gt > 0 else 0.0

    return {
        "map50": map50,
        "precision": precision,
        "recall": recall,
        "iou_matched": iou1_matched_only,
        "iou_all_gt": iou2_over_all_gt,
    }

In [ ]:
# ============================================================
# CELL: Main automated loop -- Exp x Image x Anchor Bbox -> CSV
# Now with: resume support, per-image tile caching, batched inference,
# per-image logging (time per image + running ETA)
#
# Plain English of the caching part (CHANGE 2 below): a given image
# gets processed once per GT box (once per "anchor"). Previously,
# every single anchor run re-cropped the SAME tiles from the SAME
# image from scratch -- wasted, repeated work. Now the tile crops are
# built ONCE per image (right after loading it, before the anchor
# loop starts) and reused for every anchor of that image via
# cached_tiles, then thrown away (`del cached_tiles`) once the image
# is done and we move to the next one.
# ============================================================
import psutil

done_keys = set()
file_exists = os.path.exists(OUTPUT_CSV)
if file_exists:
    existing = pd.read_csv(OUTPUT_CSV)
    existing = existing[existing["experiment_name"] == EXPERIMENT_NAME]
    done_keys = set(zip(existing["image_ID"], existing["anchor_idx"].astype(int)))
    print(f"Resuming: {len(done_keys)} rows already done for {EXPERIMENT_NAME}.")
    del existing
    gc.collect()

csv_file = open(OUTPUT_CSV, "a", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=CSV_COLUMNS)
if not file_exists:
    csv_writer.writeheader()

start_time = time.time()
n_runs = 0
image_times = []

valid_images = [rec for rec in image_records if rec[2] is not None]
n_total_images = len(valid_images)

MEM_STOP_THRESHOLD_PCT = 70  # tune based on what the [MEM] logs show

for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
    mem_pct = psutil.virtual_memory().percent
    if mem_pct > MEM_STOP_THRESHOLD_PCT:
        print(f"RAM at {mem_pct:.0f}% (threshold {MEM_STOP_THRESHOLD_PCT}%) -- "
              f"stopping cleanly before {image_id} to avoid a hard crash. "
              f"Restart the runtime and rerun this cell to resume from where you left off.")
        break

    image_t0 = time.time()

    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size
    gt_boxes = load_yolo_boxes(label_path, img_w, img_h, class_id=RUMEX_CLASS_ID)
    n_gt = len(gt_boxes)

    if n_gt == 0:
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
        continue

    # CHANGE 2: cache raw tile crops ONCE per image -- reused for every anchor below
    cached_tiles = None
    if USE_TILING:
        cached_tiles = [
            (x1, y1, x2, y2, image.crop((x1, y1, x2, y2)))
            for (x1, y1, x2, y2) in tile_bboxes(img_w, img_h, TILE_SIZE, OVERLAP)
        ]

    image_map50s = []

    for anchor_idx in range(n_gt):
        gc.collect()
        torch.cuda.empty_cache()

        mem_pct = psutil.virtual_memory().percent
        rss_gb = psutil.Process().memory_info().rss / 1e9
        if n_runs % 3 == 0:  # log at the same cadence as cleanup, keep it cheap
            print(f"        [MEM] anchor={anchor_idx} RSS={rss_gb:.2f} GB | system RAM used={mem_pct:.0f}%")
        if mem_pct > MEM_STOP_THRESHOLD_PCT:
            print(f"RAM at {mem_pct:.0f}% mid-image at {image_id} anchor={anchor_idx} -- "
                  f"stopping cleanly. Restart runtime and rerun to resume "
                  f"(this image's completed anchors are already saved; incomplete ones will retry).")
            csv_file.close()
            raise SystemExit("Stopped cleanly due to RAM threshold — restart runtime and rerun.")
        if (image_id, anchor_idx) in done_keys:
            continue  # skip BEFORE touching RNG at all

        run_t0 = time.time()

        try:

            exemplar_indices = select_exemplar_indices(n_gt, anchor_idx, N_EXEMPLARS, image_id)
            prompt_id = format_prompt_id(exemplar_indices)
            exemplar_crops = [image.crop([int(round(v)) for v in gt_boxes[i]]) for i in exemplar_indices]

            final_boxes, final_scores, final_masks = run_sam3_pipeline(
                image, exemplar_crops, use_tiling=USE_TILING,
                tile_size=TILE_SIZE, overlap=OVERLAP, threshold=THRESHOLD,
                cached_tiles=cached_tiles, batch_size=BATCH_SIZE,
                nms_iou_thresh=NMS_IOU_THRESHOLD,
            )

            metrics = compute_detection_metrics(final_boxes, final_scores, gt_boxes, iou_threshold=IOU_THRESHOLD)
            image_map50s.append(metrics["map50"])

            row = {
                "experiment_name": EXPERIMENT_NAME,
                "image_ID": image_id,
                "anchor_idx": anchor_idx,
                "Prompt_ID": prompt_id,
                "Prompt_Type": PROMPT_TYPE,
                "mAP50": metrics["map50"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "IoU1": metrics["iou_matched"],
                "IoU2": metrics["iou_all_gt"],
            }
            csv_writer.writerow(row)
            csv_file.flush()
            os.fsync(csv_file.fileno())
            n_runs += 1

            del final_boxes, final_scores, final_masks, exemplar_crops

        except Exception as e:
                print(f"    ERROR on {image_id} anchor={anchor_idx}: {type(e).__name__}: {e} -- skipping, not written to CSV.")
                gc.collect()
                torch.cuda.empty_cache()
                continue

       # gc.collect() force a CUDA sync each call, costly when batching
        gc.collect()
        torch.cuda.empty_cache()

        run_elapsed = time.time() - run_t0
        print(
            f"    [{EXPERIMENT_NAME}] run #{n_runs} | image={image_id} | "
            f"anchor={anchor_idx} ({anchor_idx+1}/{n_gt}) | prompt_id={prompt_id} | "
            f"mAP50={metrics['map50']:.3f} | time={run_elapsed:.1f}s"
        )

    # free this image's cached tiles before moving on
    del cached_tiles
    image.close()
    gc.collect()
    torch.cuda.empty_cache()

    image_elapsed = time.time() - image_t0
    image_times.append(image_elapsed)
    avg_time_per_image = np.mean(image_times)
    images_left = n_total_images - img_idx
    eta_seconds = images_left * avg_time_per_image

    map50_str = f"{np.mean(image_map50s):.3f}" if image_map50s else "N/A (all anchors already done, skipped)"

    print(
        f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} done | "
        f"{n_gt} GT box(es) | image_mAP50_mean={map50_str} | "
        f"time={image_elapsed:.1f}s | avg/image={avg_time_per_image:.1f}s | "
        f"ETA={eta_seconds/60:.1f} min ({eta_seconds/3600:.2f} h)"
    )

    rss_gb = psutil.Process().memory_info().rss / 1e9
    avail_gb = psutil.virtual_memory().available / 1e9
    print(f"    [MEM] process RSS={rss_gb:.2f} GB | system available={avail_gb:.2f} GB")

csv_file.close()
total_elapsed = time.time() - start_time
print(f"\nFinished {EXPERIMENT_NAME}: {n_runs} new rows written to {OUTPUT_CSV}")
print(f"Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} h)")

Resuming: 895 rows already done for E02_2.
        [MEM] anchor=0 RSS=1.73 GB | system RAM used=31%
[E02_2] (1/179) 20220513_Halden/DJI_20220513075116_0213 done | 1 GT box(es) | image_mAP50_mean=N/A (all anchors already done, skipped) | time=4.2s | avg/image=4.2s | ETA=12.5 min (0.21 h)
    [MEM] process RSS=1.21 GB | system available=9.87 GB
        [MEM] anchor=0 RSS=1.73 GB | system RAM used=32%
        [MEM] anchor=1 RSS=1.73 GB | system RAM used=32%
[E02_2] (2/179) 20220513_Halden/DJI_20220513075207_0234 done | 2 GT box(es) | image_mAP50_mean=N/A (all anchors already done, skipped) | time=3.7s | avg/image=4.0s | ETA=11.7 min (0.20 h)
    [MEM] process RSS=1.21 GB | system available=9.78 GB
        [MEM] anchor=0 RSS=1.73 GB | system RAM used=32%
        [MEM] anchor=1 RSS=1.73 GB | system RAM used=32%
        [MEM] anchor=2 RSS=1.73 GB | system RAM used=32%
        [MEM] anchor=3 RSS=1.73 GB | system RAM used=32%
        [MEM] anchor=4 RSS=1.73 GB | system RAM used=32%
        [ME

/tmp/ipykernel_2351/3591573759.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(mask.astype(np.uint8), mode="L")  # <-- return a PIL Image, not ndarray


    [E02_2] run #1 | image=20230426_Wallenwil/DJI_20230426111205_0193 | anchor=21 (22/48) | prompt_id=21+28+16 | mAP50=0.192 | time=30.4s
    [E02_2] run #2 | image=20230426_Wallenwil/DJI_20230426111205_0193 | anchor=22 (23/48) | prompt_id=22+42+43 | mAP50=0.192 | time=29.6s
    [E02_2] run #3 | image=20230426_Wallenwil/DJI_20230426111205_0193 | anchor=23 (24/48) | prompt_id=23+32+35 | mAP50=0.327 | time=28.4s
        [MEM] anchor=24 RSS=2.95 GB | system RAM used=40%
    [E02_2] run #4 | image=20230426_Wallenwil/DJI_20230426111205_0193 | anchor=24 (25/48) | prompt_id=24+31+17 | mAP50=0.217 | time=28.1s
    [E02_2] run #5 | image=20230426_Wallenwil/DJI_20230426111205_0193 | anchor=25 (26/48) | prompt_id=25+46+23 | mAP50=0.285 | time=28.3s
    [E02_2] run #6 | image=20230426_Wallenwil/DJI_20230426111205_0193 | anchor=26 (27/48) | prompt_id=26+43+45 | mAP50=0.273 | time=27.9s
        [MEM] anchor=27 RSS=2.98 GB | system RAM used=40%
    [E02_2] run #7 | image=20230426_Wallenwil/DJI_202304

In [ ]:
#  Aggregate statistics for ONE experiment (run separately per experiment)
import pandas as pd
import numpy as np
import os

#  Change this single line for each experiment
EXPERIMENT_CSV = "/content/drive/MyDrive/master_thesis/results/results2_E02_2.csv"
OUTPUT_DIR = "/content/drive/MyDrive/master_thesis/results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)  # just in case it doesn't exist yet

assert os.path.exists(EXPERIMENT_CSV), f"File not found: {EXPERIMENT_CSV}"
df = pd.read_csv(EXPERIMENT_CSV)

exp_name = df["experiment_name"].iloc[0]
print(f"Experiment: {exp_name}")
print(f"Total rows (image x prompt runs): {len(df)}")
print(f"Number of distinct images: {df['image_ID'].nunique()}")

Experiment: E02_2
Total rows (image x prompt runs): 850
Number of distinct images: 72


In [ ]:
#  Raw per-run stats (every row = one image + one prompt set, independent observation)
# Here, every row in our CSV is treated as one independent experiment.
#output 1 value per metric (produces one overall mean and one overall std for each metric)
raw_summary = df.agg(
    n_runs=("mAP50", "count"),
    mAP50_mean=("mAP50", "mean"),
    mAP50_std=("mAP50", "std"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    IoU1_mean=("IoU1", "mean"),
    IoU1_std=("IoU1", "std"),
    IoU2_mean=("IoU2", "mean"),
    IoU2_std=("IoU2", "std"),
)

raw_summary.to_csv(os.path.join(OUTPUT_DIR, f"raw_summary_{exp_name}.csv"), index=False)
print(f"Saved raw_summary_{exp_name}.csv to {OUTPUT_DIR}")

print(f"=== {exp_name} -- raw per-run summary (all rows independent) ===")
print(raw_summary)

Saved raw_summary_E02_2.csv to /content/drive/MyDrive/master_thesis/results/
=== E02_2 -- raw per-run summary (all rows independent) ===
                     mAP50  precision    recall      IoU1      IoU2
n_runs          850.000000        NaN       NaN       NaN       NaN
mAP50_mean        0.248138        NaN       NaN       NaN       NaN
mAP50_std         0.175447        NaN       NaN       NaN       NaN
precision_mean         NaN   0.164422       NaN       NaN       NaN
precision_std          NaN   0.090326       NaN       NaN       NaN
recall_mean            NaN        NaN  0.548219       NaN       NaN
recall_std             NaN        NaN  0.173405       NaN       NaN
IoU1_mean              NaN        NaN       NaN  0.792339       NaN
IoU1_std               NaN        NaN       NaN  0.113453       NaN
IoU2_mean              NaN        NaN       NaN       NaN  0.442014
IoU2_std               NaN        NaN       NaN       NaN  0.144195


In [ ]:
# Image-level stats (collapse multiple prompt runs per image to one value first,
# so images with more GT boxes / more prompt sets don't dominate the average)
image_level = ( # produces one mean value per image for each metric
    df.groupby("image_ID")
    .agg(
        mAP50_image_mean=("mAP50", "mean"),
        precision_image_mean=("precision", "mean"),
        recall_image_mean=("recall", "mean"),
        IoU1_image_mean=("IoU1", "mean"),
        IoU2_image_mean=("IoU2", "mean"),
        n_prompts=("mAP50", "count"),
    )
    .reset_index()
)
image_level.to_csv(os.path.join(OUTPUT_DIR, f"image_level_{exp_name}.csv"), index=False)
print(f"Saved image_level{exp_name}.csv to {OUTPUT_DIR}")

print(f"=== {exp_name} -- image-level results ({len(image_level)} images) ===")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print(image_level.to_string(index=False))

Saved image_levelE02_2.csv to /content/drive/MyDrive/master_thesis/results/
=== E02_2 -- image-level results (72 images) ===
                                    image_ID  mAP50_image_mean  precision_image_mean  recall_image_mean  IoU1_image_mean  IoU2_image_mean  n_prompts
     20220513_Halden/DJI_20220513075116_0213          0.500000              0.200000           1.000000         0.809101         0.809101          1
     20220513_Halden/DJI_20220513075207_0234          0.834983              0.023736           1.000000         0.845855         0.845855          2
     20220513_Halden/DJI_20220513075336_0270          0.554934              0.055024           0.694444         0.855907         0.595763          6
     20220513_Halden/DJI_20220513075432_0295          0.157499              0.036743           0.555556         0.786350         0.441808          3
     20220513_Halden/DJI_20220513075444_0300          0.782838              0.033345           1.000000         0.703303         0

In [ ]:
image_level_summary = pd.DataFrame({
    "experiment_name": [exp_name],
    "n_images": [image_level["image_ID"].nunique()],
    "mAP50_mean": [image_level["mAP50_image_mean"].mean()],
    "mAP50_std": [image_level["mAP50_image_mean"].std()],
    "precision_mean": [image_level["precision_image_mean"].mean()],
    "precision_std": [image_level["precision_image_mean"].std()],
    "recall_mean": [image_level["recall_image_mean"].mean()],
    "recall_std": [image_level["recall_image_mean"].std()],
    "IoU1_mean": [image_level["IoU1_image_mean"].mean()],
    "IoU2_mean": [image_level["IoU2_image_mean"].mean()],
})

image_level_summary.to_csv(os.path.join(OUTPUT_DIR, f"summary_{exp_name}.csv"), index=False)
print(f"\nSaved image_level_{exp_name}.csv and summary_{exp_name}.csv to {OUTPUT_DIR}")

print(f"\n=== {exp_name} -- image-level summary ===")
print(image_level_summary.to_string(index=False))


Saved image_level_E02_2.csv and summary_E02_2.csv to /content/drive/MyDrive/master_thesis/results/

=== E02_2 -- image-level summary ===
experiment_name  n_images  mAP50_mean  mAP50_std  precision_mean  precision_std  recall_mean  recall_std  IoU1_mean  IoU2_mean
          E02_2        72    0.357124   0.313371        0.098546       0.086263     0.623367    0.320789   0.709734   0.504732
